In [1]:
!pip install pandas
!pip install --upgrade pandas pyarrow
!pip install polars
!pip install tqdm

In [5]:
import pandas as pd
import re
DATA_ROOT = "data/feature-normalization-hackathon/data"

# Paths
TRAIN_PRODUCTS = f"{DATA_ROOT}/train/products.parquet"
TRAIN_FEATURES = f"{DATA_ROOT}/train/product_features.parquet"
VAL_PRODUCTS = f"{DATA_ROOT}/val/products.parquet"
VAL_FEATURES = f"{DATA_ROOT}/val/product_features.parquet"
TEST_PRODUCTS = f"{DATA_ROOT}/test/products.parquet"
TAXONOMY = f"{DATA_ROOT}/taxonomy/taxonomy.parquet"

# Load
train = pd.read_parquet(TRAIN_PRODUCTS)
train_feat = pd.read_parquet(TRAIN_FEATURES)
val = pd.read_parquet(VAL_PRODUCTS)
val_feat = pd.read_parquet(VAL_FEATURES)
test = pd.read_parquet(TEST_PRODUCTS)
taxonomy = pd.read_parquet(TAXONOMY)

In [6]:
# --------------------------
# 1. Normalize Text
# --------------------------
def normalize_text(text):
    if not isinstance(text, str):
        return ""
    # lowercase, remove HTML, collapse whitespace
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.lower()

train['text'] = (train['title'] + " " + train['description']).apply(normalize_text)
val['text'] = (val['title'] + " " + val['description']).apply(normalize_text)
test['text'] = (test['title'] + " " + test['description']).apply(normalize_text)

In [ ]:
from collections import defaultdict
# --------------------------
# 2. Build auto regex patterns
# --------------------------
def build_numeric_regex(train_feat, train, taxonomy):
    numeric_patterns = defaultdict(lambda: defaultdict(str))
    numeric_feats = taxonomy[taxonomy['feature_type'] == 'numeric']

    for _, row in numeric_feats.iterrows():
        cat = row['category']
        feat = row['feature_name']

        uids = train_feat[(train_feat['feature_name']==feat) & (train_feat['uid'].isin(train[train['category']==cat]['uid']))]['feature_value'].dropna().astype(str)
        if len(uids) == 0:
            continue

        escaped = [re.escape(v).replace("\\ ","\\s*") for v in uids[:50]]
        pattern = "(" + "|".join(escaped) + ")"

        numeric_patterns[cat][feat] = pattern
    return numeric_patterns

def build_categorical_regex(train_feat, train, taxonomy):
    cat_patterns = defaultdict(lambda: defaultdict(str))
    cat_feats = taxonomy[taxonomy['feature_type']=='categorical']

    for _, row in cat_feats.iterrows():
        cat = row['category']
        feat = row['feature_name']

        uids = train_feat[(train_feat['feature_name']==feat) & (train_feat['uid'].isin(train[train['category']==cat]['uid']))]['feature_value'].dropna().astype(str)
        if len(uids) == 0:
            continue

        escaped = [re.escape(v.lower()) for v in uids[:50]]
        pattern = "(" + "|".join(escaped) + ")"

        cat_patterns[cat][feat] = pattern
    return cat_patterns

numeric_patterns = build_numeric_regex(train_feat, train, taxonomy)
categorical_patterns = build_categorical_regex(train_feat, train, taxonomy)

# --------------------------
# 3. Build category -> feature map
# --------------------------
cat_features = defaultdict(list)
for _, row in taxonomy.iterrows():
    cat_features[row['category']].append({
        'feature_name': row['feature_name'],
        'feature_type': row['feature_type']
    })

# --------------------------
# 4. Feature Extraction Function
# --------------------------
def extract_features_auto(row):
    cat = row['category']
    text = row['text']
    results = []

    # Numeric features
    for feat, pattern in numeric_patterns.get(cat, {}).items():
        match = re.search(pattern, text)
        val = match.group(1) if match else None
        results.append({
            "uid": row['uid'],
            "feature_name": feat,
            "feature_value": val,
            "feature_type": "numeric"
        })

    # Categorical features
    for feat, pattern in categorical_patterns.get(cat, {}).items():
        match = re.search(pattern, text)
        val = match.group(1) if match else None
        results.append({
            "uid": row['uid'],
            "feature_name": feat,
            "feature_value": val,
            "feature_type": "categorical"
        })

    # Fallback: add missing features with None
    predicted_feats = [r['feature_name'] for r in results]
    for f in cat_features[cat]:
        if f['feature_name'] not in predicted_feats:
            results.append({
                "uid": row['uid'],
                "feature_name": f['feature_name'],
                "feature_value": None,
                "feature_type": f['feature_type']
            })

    return results

# --------------------------
# 5. Apply to Test Set
# --------------------------
submission_rows = []
for _, row in test.iterrows():
    submission_rows.extend(extract_features_auto(row))

submission = pd.DataFrame(submission_rows)
submission = submission[['uid','feature_name','feature_value','feature_type']]

submission.to_parquet("submission.parquet", index=False)
print("submission.parquet ready ✅")